# 1. Objective

Diagnose the existing weakly supervised CAM localization errors before Phase 2. This notebook reads only the saved Stage 12 reports and representative examples; it does not train a model or rerun CAM inference.

# 2. Phase 1 baseline

The official Phase 1 result remains unchanged: mean IoU 0.080087, median IoU 0.062605, IoU > 0 on 80.70% of the 601 positive test images, and IoU >= 0.5 on 0.33%. The reference paper CNN + CAM test IoU is 0.1508.

# 3. Existing CAM pipeline

Stage 8 checkpoint; Pneumonia-class (index 1) ReLU CAM; per-image min-max normalization; bilinear resize to 128 x 128; inclusive CAM >= threshold; iterative 8-connected DFS; per-image cluster-size mean +/- 2 population-SD filtering; component bounding boxes scaled to native DICOM width/height; source RSNA boxes used only after prediction; greedy one-to-one IoU matching with image score divided by max(predicted count, ground-truth count). The fixed 128 x 128 model input uses train-fitted normalization.

In [ ]:
import csv
import json
from pathlib import Path
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / "results" / "cam_threshold_analysis.csv").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "results" / "cam_threshold_analysis.csv").is_file():
    raise FileNotFoundError("Could not locate saved Stage 12 results")

with (ROOT / "results" / "cam_threshold_analysis.csv").open(newline="", encoding="utf-8") as stream:
    threshold_rows = list(csv.DictReader(stream))
with (ROOT / "results" / "cam_failure_statistics.json").open(encoding="utf-8") as stream:
    failure_stats = json.load(stream)
with (ROOT / "results" / "phase2_error_analysis" / "examples.csv").open(newline="", encoding="utf-8") as stream:
    example_rows = list(csv.DictReader(stream))

print(f"Loaded {len(threshold_rows)} threshold settings and {len(example_rows)} indexed examples.")

# 4. Threshold sensitivity

All settings reuse one CAM per fixed test image and preserve the existing post-processing and metric definitions. The best value below is an exploratory test-set result, not a replacement Phase 1 result.

In [ ]:
header = "| Threshold | Mean IoU | Median IoU | IoU > 0 | IoU >= 0.5 | No predicted box |"
separator = "|---:|---:|---:|---:|---:|---:|"
lines = [header, separator]
for row in threshold_rows:
    lines.append(
        f"| {float(row['threshold']):.2f} | {float(row['mean_iou']):.6f} | "
        f"{float(row['median_iou']):.6f} | {float(row['percentage_iou_gt_0']):.2f}% | "
        f"{float(row['percentage_iou_ge_0_5']):.2f}% | {int(row['num_images_with_no_predicted_boxes'])} |"
    )
display(Markdown("\n".join(lines)))
best = max(threshold_rows, key=lambda row: float(row["mean_iou"]))
print(f"Best grid threshold by mean IoU: {float(best['threshold']):.2f}")

# 5. Representative successes

The indexed examples include a high-overlap case and a partial-overlap case. Their image panels are saved under `results/phase2_error_analysis/`; no large visualization grid is generated by this notebook.

In [ ]:
success_types = {"good_localization", "partial_localization"}
for row in example_rows:
    if success_types.intersection(row["case_types"].split(";")):
        print(row["case_types"], row["image_id"], "IoU", row["iou"], row["visualization"])

# 6. Representative failures

The saved examples cover large false-positive regions, small predictions, multiple disconnected components, anatomically irrelevant activation, no useful activation, and missed ground-truth regions. The index records each case's image-level IoU and box counts.

In [ ]:
failure_terms = (
    "false_positive", "small_predicted", "multiple_disconnected",
    "irrelevant_anatomical", "no_useful_activation", "ground_truth_region_missed"
)
for row in example_rows:
    if any(term in row["case_types"] for term in failure_terms):
        print(row["case_types"], row["image_id"], "IoU", row["iou"], row["visualization"])

# 7. Failure statistics

Areas are measured in pixels squared at native DICOM resolution. Predicted-to-ground-truth ratios use per-image total box areas, including images with no predicted box.

In [ ]:
print(json.dumps(failure_stats, indent=2))

# 8. Findings

Threshold 0.30 gave the best mean IoU in the evaluated grid (0.087996), but only a small improvement over the official 0.50 result and no meaningful improvement in the fraction reaching IoU >= 0.5. Most images produced one box, yet the typical predicted area was larger than ground truth and reviewed maps sometimes focused below/outside annotated pulmonary regions. The dominant issue is spatial precision and relevance, not a lack of any activation.

# 9. Proposed Phase 2 improvement

Evaluate Grad-CAM++ with the existing frozen Stage 8 checkpoint. It uses gradient-weighted class-specific activations, remains weakly supervised, requires no bounding-box labels for training, and is clearly distinct from the baseline classifier-weighted CAM. Choose any operating threshold using validation data and evaluate the selected setting once on the unchanged test split. See `PHASE2_PLAN.md` for the controlled protocol and success criteria.